# Config

In [7]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [8]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [23]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
#filepath = os.path.join(path, "datasets/features.csv")
filepath = os.path.join(path, "datasets/data_translated.csv")
df=pd.read_csv(filepath)

feat_col = "Desafío País"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Desafío País"].notna()]

#Eliminar duplicados
df = df.drop_duplicates("Código VRID")

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Desafío País"])
savepath = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)


Test size: 247
Fold 0 - Val size: 330
Archivo guardado exitosamente en /tmp/final_project/dataSplits/desafios/train_test_ids_3folds.json


# 1) TF-IDF

In [24]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [ ]:
"""
path2 = "/tmp/desafios/new_data"
filepath=os.path.join(path2, "new_data_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path2, "data_translated.csv")
df = pd.read_csv(filepath)
"""

In [25]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [28]:
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)

split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
#split_idx_path = os.path.join(path2, "new_data_ids_3folds.json")

savepath = os.path.join(path, "output/desafios/TF_IDF")
model_paths = ["1", "2", "3", "4"]
for idx, des in enumerate(comb_desafios):
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    #Creacion de vectores TFID
    X_train, X_test, vectorizer = gen_TFID_vectors(X_train, X_test, return_vectorizer=True)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds

    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results_test, preds_test, save_preds=True, mode_classification="binary")
    

(987, 17354) (247, 17354)
📊 train: Counter({0: 887, 1: 100})
📊 test: Counter({0: 222, 1: 25})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.6, 'std_test_score': 0.04}
XGBClassifier: {'mean_test_score': 0.69, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.67, 'std_test_score': 0.05}
LogisticRegression
{'accuracy': 0.8744939271255061, 'f1_macro': 0.6828743010975358, 'cm': array([[204,  18],
       [ 13,  12]]), 'precision': 0.4, 'recall': 0.48, 'f1_es': 0.8765227021040976, 'f1_en': 0.8618986091054192, 'cm_es': array([[99, 15],
       [ 3, 12]]), 'cm_en': array([[105,   3],
       [ 10,   0]])}
RandomForestClassifier
{'accuracy': 0.8947368421052632, 'f1_macro': 0.6756565656565656, 'cm': array([[212,  10],
       [ 16,   9]]), 'precision': 0.47368421052631576, 'recall': 0.36, 'f1_es': 0.882157

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 707, 1: 280})
📊 test: Counter({0: 176, 1: 71})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.72, 'std_test_score': 0.03}
RandomForestClassifier: {'mean_test_score': 0.7, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.68, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.72, 'std_test_score': 0.02}
LogisticRegression
{'accuracy': 0.8016194331983806, 'f1_macro': 0.7376043360433604, 'cm': array([[160,  16],
       [ 33,  38]]), 'precision': 0.7037037037037037, 'recall': 0.5352112676056338, 'f1_es': 0.8673888817844682, 'f1_en': 0.7098229122157338, 'cm_es': array([[92,  8],
       [ 9, 20]]), 'cm_en': array([[68,  8],
       [24, 18]])}
RandomForestClassifier
{'accuracy': 0.7854251012145749, 'f1_macro': 0.6996443730641276, 'cm': array([[163,  13],
 

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 698, 1: 289})
📊 test: Counter({0: 174, 1: 73})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.05}
SVC: {'mean_test_score': 0.69, 'std_test_score': 0.02}
LogisticRegression
{'accuracy': 0.7165991902834008, 'f1_macro': 0.6511299435028248, 'cm': array([[142,  32],
       [ 38,  35]]), 'precision': 0.5223880597014925, 'recall': 0.4794520547945205, 'f1_es': 0.7553523212263741, 'f1_en': 0.660278633795583, 'cm_es': array([[73, 18],
       [14, 24]]), 'cm_en': array([[69, 14],
       [24, 11]])}
RandomForestClassifier
{'accuracy': 0.708502024291498, 'f1_macro': 0.6123801220575413, 'cm': array([[149,  25],
  

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 738, 1: 249})
📊 test: Counter({0: 185, 1: 62})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.57, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.64, 'std_test_score': 0.03}
LogisticRegression
{'accuracy': 0.631578947368421, 'f1_macro': 0.5444928564190901, 'cm': array([[132,  53],
       [ 38,  24]]), 'precision': 0.3116883116883117, 'recall': 0.3870967741935484, 'f1_es': 0.7692806922660899, 'f1_en': 0.4807102502017756, 'cm_es': array([[94,  8],
       [19,  8]]), 'cm_en': array([[38, 45],
       [19, 16]])}
RandomForestClassifier
{'accuracy': 0.5910931174089069, 'f1_macro': 0.5376160732489389, 'cm': array([[115,  70],
 

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [31]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/desafios/TF_IDF/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: SVC
Prediction: [0]


# 2) SPECTER

In [18]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [19]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [29]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline
from models.specter import BERT_vectorizer

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
#Guardado de resultados
savepath = os.path.join(path, "output/desafios/SPECTER")
model_paths = ["1", "2", "3", "4"]
for des in comb_desafios:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    # Calcular embeddings
    # Parámetros para cargar modelo
    BASE_MODEL = "allenai/specter2_base"
    ADAPTER_NAME="allenai/specter2_classification"
    X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
    X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds
    
    vectorizer = BERT_vectorizer(BASE_MODEL, ADAPTER_NAME)
    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results_test, preds_test, save_preds=True, mode_classification="binary")

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(987, 768) (247, 768)
📊 train: Counter({0: 738, 1: 249})
📊 test: Counter({0: 185, 1: 62})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.58, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.0}
XGBClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.61, 'std_test_score': 0.01}
LogisticRegression
{'accuracy': 0.6356275303643725, 'f1_macro': 0.5986205402282248, 'cm': array([[116,  69],
       [ 21,  41]]), 'precision': 0.37272727272727274, 'recall': 0.6612903225806451, 'f1_es': 0.7946436727013646, 'f1_en': 0.4712814485039312, 'cm_es': array([[87, 15],
       [12, 15]]), 'cm_en': array([[29, 54],
       [ 9, 26]])}
RandomForestClassifier
{'accuracy': 0.6477732793522267, 'f1_macro': 0.5436126154826377, 'cm': array([[139,  46],
       [ 41,  21]]), 'preci

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [30]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/desafios/SPECTER/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: LogisticRegression


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
Prediction: [0]
